In [3]:
import pandas as pd
import os

csv_path = "../../../../../results/results.csv"
latex_out = "tables/vrp_results.tex"
df = pd.read_csv(csv_path)

In [4]:
def agent_type_name(t):
    t = str(t).upper()
    if t in ["GROUND", "G"]:
        return "G"
    elif t in ["AERIAL", "A"]:
        return "A"
    else:
        return t

def method_type_name(t):
    t = str(t)
    if "het" in t:
        return "Het"
    elif "hom" in t:
        return "Hom"
    else:
        return t

In [5]:
df["agent_type_norm"] = df["agent_type"].apply(agent_type_name)
df["method_norm"] = df["method"].apply(method_type_name)

In [6]:
# Count how many of each agent type per experiment
agent_count = {}
# Choose one baseline to avoid doubles
baseline = df["method_norm"].unique()[0]
for exp_id, agent_type, method in zip(df["exp_id"], df["agent_type_norm"], df["method_norm"]):
    if exp_id not in agent_count:
        agent_count[exp_id] = ""
    if  method == baseline:
        agent_count[exp_id] += f"{agent_type},"

# Add agent_config to each experiment
df["agent_config"] = [agent_count[exp_id] for exp_id in df["exp_id"]]

In [8]:
agg_cols = ["total_cost", "distance", "time", "traversability", "collision", "safety"]

# Number of significant decimals for each metric
decimals_avg_max = [0, 0, 0, 3, 3, 3]  # for mean, min, max
decimals_std = [0, 0, 0, 2, 2, 2]      # for std

# Aggregate mean, std, min, max
summary = (
    df.groupby(["map", "method_norm", "agent_config"])[agg_cols]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

# Flatten MultiIndex columns
summary.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] for col in summary.columns.values]

# --- Round numeric columns per metric ---
for i, m in enumerate(agg_cols):
    for suffix, dec in zip(["mean", "std", "min", "max"], [decimals_avg_max[i], decimals_std[i], decimals_avg_max[i], decimals_avg_max[i]]):
        col_name = f"{m}_{suffix}"
        if dec == 0:
            summary[col_name] = summary[col_name].round(0).astype(int)
        else:
            summary[col_name] = summary[col_name].round(dec)

# --- Create LaTeX-formatted mean ± std strings ---
# for i, m in enumerate(agg_cols):
#     summary[f"{m}_latex"] = summary.apply(
#         lambda row: f"${row[f'{m}_mean']} \\pm {row[f'{m}_std']} ({row[f'{m}_max']})$",
#         axis=1
#     )

for i, m in enumerate(agg_cols):
    summary[f"{m}_latex"] = summary.apply(
        lambda row: f"${row[f'{m}_max']}$",
        axis=1
    )

# --- Keep only relevant columns ---
cols = ["map", "method_norm", "agent_config"] + [f"{m}_latex" for m in agg_cols]
df_disp = summary[cols].copy()

# --- Sort for consistent order ---
df_disp = df_disp.sort_values(["map", "method_norm", "agent_config"]).reset_index(drop=True)

# --- Produce LaTeX-friendly row outputs per metric ---
for metric in agg_cols:
    print(f"\n% ===== {metric} =====")
    
    # Method order (reversed from appearance)
    method_order = df_disp["method_norm"].unique()[::-1]
    
    pivot = (
        df_disp.pivot_table(
            index=["method_norm"],
            columns=["map", "agent_config"],
            values=f"{metric}_latex",
            aggfunc="first"
        )
        .reindex(method_order)
    )
    
    # Print LaTeX rows
    for method, row in pivot.iterrows():
        values = " & ".join(row.fillna("").tolist())
        print(f"{metric} & {method} & {values} \\\\")



% ===== total_cost =====
total_cost & Hom & $15215$ & $30963$ & $13219$ & $24048$ & $16555$ & $33771$ & $16258$ & $31656$ \\
total_cost & Het & $23126$ & $48302$ & $19055$ & $36580$ & $25815$ & $49098$ & $25048$ & $56045$ \\

% ===== distance =====
distance & Hom & $653$ & $1328$ & $45$ & $82$ & $44$ & $89$ & $75$ & $146$ \\
distance & Het & $980$ & $2040$ & $64$ & $117$ & $61$ & $117$ & $110$ & $233$ \\

% ===== time =====
time & Hom & $1305$ & $2655$ & $90$ & $163$ & $88$ & $179$ & $150$ & $293$ \\
time & Het & $980$ & $2064$ & $64$ & $117$ & $68$ & $130$ & $116$ & $257$ \\

% ===== traversability =====
traversability & Hom & $0.998$ & $0.847$ & $0.243$ & $0.442$ & $0.342$ & $0.567$ & $0.716$ & $0.875$ \\
traversability & Het & $0.005$ & $0.01$ & $0.074$ & $0.083$ & $0.0$ & $0.001$ & $0.068$ & $0.256$ \\

% ===== collision =====
collision & Hom & $0.155$ & $0.466$ & $0.028$ & $0.096$ & $0.169$ & $0.191$ & $0.057$ & $0.118$ \\
collision & Het & $0.028$ & $0.052$ & $0.007$ & $0.022$ &